# Lecture Analyzer — Kaggle runner
Video (Hebrew/Russian, handwritten board) → Whisper transcript → keyframes → VL report → **one TaskSpec JSON per task class**.

**Before running:** Settings → Accelerator = **GPU T4**, and Internet = **On** (needed for deps + model download + Google Drive).

Set the source in `config.yaml` (a Kaggle-dataset path **or** a Google Drive id). For the Claude backend, add `ANTHROPIC_API_KEY` as a Kaggle Secret.

In [ ]:
# 1. Get the code + deps
# Option A: clone your repo (replace URL). Option B: upload the repo as a Kaggle dataset.
# !git clone https://github.com/<you>/lection-analyzer.git
# %cd lection-analyzer
!pip install -q faster-whisper scenedetect opencv-python-headless imagehash gdown pydantic pyyaml qwen-vl-utils
!pip install -q -U transformers accelerate bitsandbytes anthropic
import sys; sys.path.insert(0, 'src')

In [ ]:
# 2. (Optional) Claude API key from Kaggle Secrets — only if a backend is set to 'claude'.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['ANTHROPIC_API_KEY'] = UserSecretsClient().get_secret('ANTHROPIC_API_KEY')
    print('Claude key loaded')
except Exception as e:
    print('No Claude key (fine if using local backend):', e)

In [ ]:
# 3. Edit config.yaml for this run (lecture id + source). Example shown; uncomment to apply.
import yaml
cfg = yaml.safe_load(open('config.yaml'))
cfg['lecture'] = 'lec01'
# cfg['ingest']['gdrive_id'] = 'PUT_DRIVE_FILE_ID_HERE'
# cfg['ingest']['source_path'] = '/kaggle/input/my-lectures/lec01.mp4'
# cfg['backends']['vl_backend'] = 'claude'   # flip if local handwriting/tables are poor
yaml.safe_dump(cfg, open('config.yaml', 'w'), allow_unicode=True)
print(cfg['lecture'], cfg['ingest'], cfg['backends'])

In [ ]:
# 4a. Run CPU-light stages first (no model in VRAM yet).
from lection_analyzer.config import load_config
from lection_analyzer import ingest, transcribe, keyframes
cfg = load_config('config.yaml')
ingest.run(cfg)
transcript = transcribe.run(cfg)   # Whisper loads, transcribes, then we free it below
index = keyframes.run(cfg, transcript)
print('frames:', len(index.frames))

In [ ]:
# 4b. Free any GPU memory before loading the VL model (stages run one model at a time).
import gc, torch
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 4c. Model stages: VL report + synthesis. Builds the configured backend(s).
from lection_analyzer.backends import build_backends
from lection_analyzer import vl_report, synthesize
vl, llm = build_backends(cfg.backends)
report = vl_report.run(cfg, index, vl)
specs = synthesize.run(cfg, report, transcript, llm)
print('task classes:', [s.task_class for s in specs])

In [ ]:
# 5. Inspect outputs. These .json files are what you load into your Agent.
import glob
for f in sorted(glob.glob(str(cfg.output_dir / '*.md'))):
    print('\n' + '=' * 70 + '\n' + open(f).read())

Download the `output/<lecture>/` folder (it's in `/kaggle/working`). Each `*.json` is a TaskSpec ready to register in your Agent.

**Re-run a single stage:** delete its artifact under `data/<lecture>/` (e.g. `vl_report.json`) and re-run, or use `python -m lection_analyzer.pipeline --only vl_report`.